# RoboCode paper results: Tables I and II

Loads saved experiment results and generates the paper table LaTeX. Configure data paths below; see `README.md` for the required result archives and audited PDDLStream reevaluation export. No agent or environment evaluation is run.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import importlib, results_lib as rl

importlib.reload(rl)
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)


## Configuration

`ROOTS` are searched recursively; later roots never override earlier ones for the same (experiment, replicate). Set `DRIVE_FOLDER_ID` to also sync `.zip` bundles from the shared Drive through the `robocode-drive` rclone remote (archives under a path containing `Outdated` are skipped).

In [ ]:
# Paths are relative to notebooks/; edit them for your downloaded data.
DATA = Path("../paper_data")
ROOTS = {
    "final_results": DATA / "final_results",
    "drive_cache": DATA / "drive_cache/runs",
    "drive_staging": DATA / "drive_staging/runs",
    "genplan_drive": DATA / "genplan/runs",
}
PDDLSTREAM_REEVALUATION = DATA / "pddlstream-reevaluation"
OVERLEAF = None  # Optional Path to a local paper checkout.
DRIVE_FOLDER_ID = None  # Optional shared Drive folder; requires configured rclone.
DRIVE_CACHE = DATA / "drive_downloads"
PREFER_RESCORE = False
OUT = Path.cwd() / "out"
OUT.mkdir(exist_ok=True)


In [ ]:
if DRIVE_FOLDER_ID:
    ROOTS["drive"] = rl.sync_drive(DRIVE_FOLDER_ID, DRIVE_CACHE)
if not any(root.is_dir() for root in ROOTS.values()):
    raise FileNotFoundError(
        "Configure ROOTS with your downloaded experiment results; see README.md"
    )
if not (PDDLSTREAM_REEVALUATION / "manifest.json").is_file():
    raise FileNotFoundError(
        "Configure PDDLSTREAM_REEVALUATION with the audited export; see README.md"
    )
runs = rl.load_runs(ROOTS)
# Dynamic 3D: the September 3-7 sweeps ran on defective environments (see the dynamic-3D audit), so only
# the re-run campaign counts; runs from those batch folders and the Drive caches are dropped for that family
# before de-duplication, since a re-run can share its Experiment ID with the old sweep.
OLD_DYNAMIC3D_BATCHES = (
    "strict_blackbox_2026-09-03",
    "whitebox_2026-09-05",
    "planner_baselines_2026-09-06",
    "planner_baselines_2026-09-07",
)
runs = rl.drop_old_dynamic3d(
    runs, OLD_DYNAMIC3D_BATCHES, ("drive_cache", "drive_staging")
)
# Runs whose score measured a harness failure rather than the agent, keyed by run directory so a
# re-launch of the same replicate is kept; the reason stays next to each entry.
EXCLUDED_RUNS = {
    "pr2blocked_generalized__agentic__none__blackbox__strict__codex_gpt56sol__timeout_60s__28cf8f22/2026-09-10_01-19-14/replicate_222": "sandbox died after two turns ($3, 'Agent stopped early' with apply_patch and exec errors); 0.00 is not a policy score",
}
runs = rl.exclude_runs(runs, EXCLUDED_RUNS)
df = rl.to_frame(runs, prefer_rescore=PREFER_RESCORE)
# Paired success and timing from the complete, audited sweep; original JSON files stay intact.
df = rl.apply_planner_reevaluation(df, PDDLSTREAM_REEVALUATION)
print(len(runs), "runs found,", len(df), "after de-duplication")
df["column"].value_counts(dropna=False)


## Sanity checks

Every run should share the evaluation seed and evaluate 100 episodes; `timed_out` counts episodes killed at the 60 s per-instance limit; the harness records them with zero steps whatever happened before the kill.

In [ ]:
print("eval seeds:", df["eval_seed"].unique())
print("episodes per run:", df["n_episodes"].value_counts().to_dict())
df[df["timed_out"] > 0][
    ["experiment_id", "replicate", "solve_rate", "timed_out", "rescore", "source"]
]


## Table 1

In [ ]:
rl.table1_frame(df)


### Tables 1 and 2, paper version

The two blocks that `sections/experiments.tex` of the Overleaf clone inputs from `tables/table-main.tex` and `tables/table-pending.tex`: environments as columns, methods as rows (model name on the tiny line under the method), mean over five runs with `[min--max]` across runs on the tiny row below. Bold marks the best mean and, in the range row, the best max per environment among the main-setting rows; the `+ source` row is a different access setting and is never bolded. Cells with fewer than five finished runs stay blank. Table 2 lists the Dynamic 3D environments (re-run campaign only, see the configuration cell; Dynamo shows the re-evaluations on the fixed environment, the invalidated runs sit in `robocode_final_results_quarantine/`) and the PDDLStream environments; both tables span two columns, with captions below the tables. Codex precedes Claude Code Opus 5, which is immediately above the source-access row. The older Claude model row is left out of both tables. The check cell reports whether the clone's table files still match; set `WRITE_OVERLEAF = True` there to overwrite them (then compile, review, commit and push).

The formatting matches the September 13 paper layout. The local data roots do not yet include the paper's one-shot evaluations or its latest Dynamic 3D/PDDLStream GenPlan results, so numerical equality is not currently expected. Keep `WRITE_OVERLEAF = False` until those data are synced and the differences reviewed.


In [ ]:
HEAD = "\\fontsize{6}{6.5}\\selectfont "
CAPTION_1 = (
    "\\textbf{Success rate over environments}. We report the mean over five runs, with [min--max] across "
    "run-level success rates below. Bold marks the best mean and the best max per environment among the "
    "main-setting rows. "
    "Claude Code (CC) + source additionally gives the agent the environment source code. A dash marks "
    "environments for which KinDER provides no planner. Dynamic 3D and PDDLStream environments are listed "
    "in Table~\\ref{tab:main-2}."
)
CAPTION_2 = (
    "\\textbf{Dynamic 3D and PDDLStream environments}. Same protocol and notation as Table~\\ref{tab:main}. "
    "The planner for the PDDLStream environments is PDDLStream itself."
)
PAPER_COLUMNS = [
    c for c in rl.COLUMNS if c != "cc_older"
]  # no runs planned for the older Claude model
PAPER_COLUMNS.remove("cc_opus5")
PAPER_COLUMNS.insert(PAPER_COLUMNS.index("source"), "cc_opus5")
MODELS = {
    "genplan": "Opus 5",
    "cc_opus5": "Opus 5",
    "codex": "GPT-5.6 Sol",
    "source": "Opus 5",
}  # tiny line under each method
STYLE = dict(
    stacked=True,
    ranges=True,
    angle=30,
    header_size=HEAD,
    colsep="0.8pt",
    short_labels=True,
    fit=True,
    columns=PAPER_COLUMNS,
    sublabels=MODELS,
)
paper1 = rl.table1_latex_transposed(
    df,
    families=("Kinematic 2D", "Dynamic 2D", "Kinematic 3D"),
    caption=CAPTION_1,
    label="tab:main",
    bold_best=True,
    bold_best_max=True,
    **STYLE
)
paper2 = rl.table1_latex_transposed(
    df,
    families=("Dynamic 3D", "PDDLStream"),
    caption=CAPTION_2,
    label="tab:main-2",
    all_envs=True,
    bold_best=True,
    bold_best_max=True,
    single_column=False,
    **STYLE
)
(OUT / "table1_paper.tex").write_text(paper1 + "\n")
(OUT / "table2_paper.tex").write_text(paper2 + "\n")
print(paper1)
print()
print(paper2)


In [ ]:
WRITE_OVERLEAF = False  # True overwrites the clone's table files with the blocks above
TABLE_FILES = {
    "tables/table-main.tex": paper1 + "\n",
    "tables/table-pending.tex": paper2 + "\n",
}
if OVERLEAF is not None and OVERLEAF.exists():
    for rel, body in TABLE_FILES.items():
        path = OVERLEAF / rel
        if WRITE_OVERLEAF:
            path.write_text(body)
        print(
            f"{rel} matches this notebook's output:",
            path.exists() and path.read_text() == body,
        )


Cells with fewer than five replicates (still running) are blank above; show them with a superscript count instead:

In [ ]:
print(rl.table1_latex(df, show_partial=True))
